<H2>AUTO DESCARGA SURVEY<H2>

In [16]:
import pandas as pd
import pyautogui as robot
import time
import datetime
from selenium import webdriver
from selenium.webdriver.support.ui import WebDriverWait 
from selenium.webdriver.support import expected_conditions as EC 
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.common.action_chains import ActionChains
from selenium.common.exceptions import TimeoutException
import time
import pandas as pd
import re
import math
import os
import glob
import shutil
import sys
from dotenv import load_dotenv

In [2]:
def esperar_invisibilidad(driver, ruta, timeout=15):
    try:
        WebDriverWait(driver, timeout).until(EC.invisibility_of_element_located((By.XPATH, ruta)))
        return True
    
    except TimeoutException:
        print(f"Demasiada espera.")
        return True 

In [17]:
#DEJADO DE USAR
def inicio_sesion_robot():
    load_dotenv(dotenv_path="credenciales.env")
    usuario_arca = os.getenv('usuario_arca')
    password = os.getenv('contraseña_arca')
    robot.press('esc')
    time.sleep(2)
    
    for i in range (0,6):
        robot.press('tab') #6veces

    robot.write(usuario_arca, interval=0.1)
    time.sleep(2)
    robot.press('tab')
    robot.write(password, interval=0.1)
    time.sleep(2)
    robot.press('tab')
    time.sleep(2)
    robot.press('tab')
    time.sleep(2)
    robot.press('enter')

In [18]:
def tipo_elemento(driver,ruta,elemento):
    diccionario = {
        'clickable': WebDriverWait(driver, 60).until(EC.element_to_be_clickable((By.XPATH, ruta))),
        'existente': WebDriverWait(driver, 60).until(EC.presence_of_element_located((By.XPATH, ruta)))
        
    }

    return diccionario[elemento]

In [19]:
def tipo_elemento_css(driver,ruta,elemento):
    diccionario = {
        'css': WebDriverWait(driver,120).until(EC.presence_of_element_located((By.CSS_SELECTOR,ruta)))
    }

    return diccionario[elemento]

In [20]:
def inicio_sesion(driver):
    div_usuario  = '/html/body/div/div/div[3]/div/form/div/div[2]/div[1]/div/input'
    div_password = '/html/body/div/div/div[3]/div/form/div/div[2]/div[2]/div/input'
    div_login    = '/html/body/div/div/div[3]/div/form/div/div[3]/div[1]/button'
    
    load_dotenv(dotenv_path="credenciales.env")
    usuario_arca = os.getenv('usuario_arca')
    password = os.getenv('contraseña_arca')


    time.sleep(5)
    input_usuario = tipo_elemento(driver,div_usuario,'existente')
    input_password = tipo_elemento(driver,div_password,'existente')
    button_login = tipo_elemento(driver,div_login,'existente')
    time.sleep(2)
    input_usuario.send_keys(usuario_arca)
    time.sleep(2)
    input_password.send_keys(password)
    time.sleep(2)
    button_login.click()


In [ ]:
#def click_survey():
#    time.sleep(5)
#    robot.click(x=200, y=615)
#    time.sleep(5)
#    robot.click(x=200, y=750)

In [37]:
def click_survey(driver):
    xpath_mng = "//div[contains(@data-menu-id, '/SurveyManagement')]"
    xpath_rvw = "//li[contains(@data-menu-id, '/SurveyManagement/SurveyReview')]"

    boton_survey_mng = tipo_elemento(driver, xpath_mng, 'clickable')
    boton_survey_mng.click()

    time.sleep(1)

    boton_survey_rvw = tipo_elemento(driver, xpath_rvw, 'clickable')
    boton_survey_rvw.click()



In [22]:
def opciones_div(valor, opcion):
    diccionario = {'fecha':f'/html/body/div/div/div[2]/div[1]/div/div[2]/div[2]/div/div[2]/div[1]/div[1]/div/div[2]/div[{valor}]/div/input',
                   'org-store':f'/html/body/div/div/div[2]/div[1]/div/div[2]/div[2]/div/div[2]/div[1]/div[1]/div/div[2]/div[{valor}]/div[1]/span/span[1]/input'}
    
    return diccionario[opcion]

In [23]:
def scroll(driver):
    ultimo_tamanio = 0
    while True:
        driver.execute_script(
            "arguments[0].scrollTop = arguments[0].scrollHeight",
            driver.find_element(By.CLASS_NAME, "ag-body-viewport")
        )
        time.sleep(1)
        nuevo_tamanio = driver.execute_script(
            "return arguments[0].scrollTop",
            driver.find_element(By.CLASS_NAME, "ag-body-viewport")
        )
        if nuevo_tamanio == ultimo_tamanio:
            break
        ultimo_tamanio = nuevo_tamanio


In [24]:
def subir_excel(archivos_encontrados, patron):
    if not archivos_encontrados:
        print(f"No se encontró ningún archivo que coincida con el patrón: {patron}")
    else:
        archivo_a_cargar = archivos_encontrados[0]
        print(f" Archivo encontrado: {archivo_a_cargar}")
    
        try:
            df = pd.read_excel(archivo_a_cargar)
            print("Archivo cargado exitosamente.")
        
        except Exception as e:
            print(f"Error al intentar cargar el archivo: {e}")
    
    return df

In [25]:
def check_mediciones(ruta_descarga, p_clave = "Survey Review"):
    band = False
    try:
        lista_archivos = os.listdir(ruta_descarga)
        for archivo in lista_archivos:
            if p_clave in archivo:
                band = True
                return band

    except Exception as e:
        print("No existe carpeta") 
    return band


In [ ]:
ruta_descarga = r'C:\Users\bbartolome\Downloads'
autoservicios = r"\AUTOSERVICIOS"
cstores       = r"\CSTORES"

options = webdriver.ChromeOptions()
options.add_argument('--start-maximized')
options.add_argument('--disable-extensions')

load_dotenv(dotenv_path='credenciales.env')
url_web = os.getenv('ruta_web')

options.add_experimental_option("prefs", {
    "download.default_directory": ruta_descarga + cstores,
    "download.prompt_for_download": False,
    "download.directory_upgrade": True,
    "safebrowsing.enabled": True
})

driver_path    = r'C:\Users\bbartolome\Downloads\Selenium\chromedriver.exe'

driver = webdriver.Chrome(options=options)
driver.get(url_web)

fechas   = ['12/05/2025', '12/07/2025'] #"mm/dd/yyyy"
opciones = ['','C-STORE']

div_search       = '/html/body/div[1]/div/div[2]/div[1]/div/div[2]/div[2]/div/div[2]/div[1]/div[1]/div/div[2]/button[1]'
div_export_menu  = '/html/body/div[1]/div/div[2]/div[1]/div/div[2]/div[2]/div/div[2]/div[1]/div[1]/div/div[2]/button[3]'
div_export       = '/html/body/div/div/div[1]/div[1]/div[5]/div/button[2]'
pag_carga        = "//div[contains(@class, 'ant-modal-content')]"
div_filtro       = "/html/body/div[1]/div/div[2]/div[1]/div/div[2]/div[2]/div/div[2]/div[1]/div[2]/div/div[3]/div[1]/div[2]/div[1]/div[2]/div/div/div[1]/div[2]/div/span/span"
div_input_filtro = '/html/body/div[1]/div/div[2]/div[1]/div/div[2]/div[2]/div/div[2]/div[1]/div[2]/div[1]/div[3]/div[3]/div/div[3]/div/div/div/div/div[1]/div/input[1]'
div_excel        = '/html/body/div[5]/div/ul/li[1]'


inicio_sesion(driver)
time.sleep(1)
esperar_invisibilidad(driver, pag_carga, timeout=20)
time.sleep(1)
click_survey(driver)

#opciones cabecera (fechas, c-store, organizacion)
for i in range(2,9):

    if i == 2 or i == 8:
        div_org_c_store = opciones_div(i,'org-store')
        input_org_c_tore = tipo_elemento(driver,div_org_c_store,'clickable')
        input_org_c_tore.send_keys(opciones[math.floor(math.sqrt(i))-1])
        time.sleep(1)
        input_org_c_tore.send_keys(Keys.ENTER)
        time.sleep(2)

    elif i == 4 or i == 5:
        div_fechas = opciones_div(i, 'fecha')
        input_fecha = tipo_elemento(driver, div_fechas,'clickable')
        input_fecha.send_keys(Keys.CONTROL + 'a')
        time.sleep(1)
        input_fecha.send_keys(Keys.BACKSPACE)
        time.sleep(1)
        input_fecha.send_keys(fechas[i - 4])
        input_fecha.send_keys(Keys.ENTER)
        time.sleep(2)
    
    else:
        continue

#click al botón search
button_search = tipo_elemento(driver, div_search,'clickable')
button_search.click()
time.sleep(10)

#verificamos que la lista de mediciones esté
ruta_descargas_carpetas = ruta_descarga + cstores
export = check_mediciones(ruta_descargas_carpetas)

if not export:

    #click al botón exportar
    button_export_menu = tipo_elemento(driver, div_export_menu,'clickable')
    button_export_menu.click()
    time.sleep(1)

    #click opcion excel
    button_excel = tipo_elemento(driver,div_excel,'existente')
    button_excel.click()
    time.sleep(15)

else:
    print("Ya existe lista de mediciones")

#Control de descarga
patron = os.path.join(ruta_descargas_carpetas, 'Survey*.XLSX')
archivos_encontrados = glob.glob(patron)
Dataframe = subir_excel(archivos_encontrados, patron)
Dataframe_validos = Dataframe[Dataframe['Session Review Status'] != 'Reject'].reset_index(drop=True)


#Proceso de busqueda, filtrado y descarga
for i in range(0,len(Dataframe_validos)):

    #Buscar el div de filtro
    filtro = driver.find_element(By.XPATH,div_filtro)
    driver.execute_script("arguments[0].click();", filtro)
    time.sleep(1)

    #Copia Id Session
    input_id_session = tipo_elemento(driver,div_input_filtro,'clickable')
    time.sleep(1)
    input_id_session.send_keys(Keys.CONTROL + 'a')
    time.sleep(1)
    input_id_session.send_keys(Keys.DELETE)
    time.sleep(1)
    input_id_session.send_keys(Dataframe_validos['Session Uid'][i])
    time.sleep(2)
    input_id_session.send_keys(Keys.ENTER)
    time.sleep(3)

    try:
        #Esperemos que aparezca la tabla
        div_fila_css = "div.ag-row[row-index='0']"
        fila_aparece = tipo_elemento_css(driver,div_fila_css,'css')

        #Buscamos las filas de la tabla
        filas = driver.find_elements(By.CSS_SELECTOR, div_fila_css)
        cod_ventana_principal = driver.current_window_handle
        ActionChains(driver).double_click(filas[0]).perform()
        time.sleep(2)

    except Exception as e:
        print("No cargó la tabla a tiempo, o no aparecio ningun dato en esta session ")
        continue

    try:
        cod_ventanas = driver.window_handles
        nueva_ventana = [h for h in cod_ventanas if h != cod_ventana_principal][0]
        driver.switch_to.window(nueva_ventana)

        esperar_invisibilidad(driver, pag_carga, timeout=60)
        time.sleep(3)
        button_export = tipo_elemento(driver, div_export,'clickable')
        button_export.click()
        time.sleep(3)
        esperar_invisibilidad(driver, pag_carga, timeout=60)
        time.sleep(3)

        driver.close()
        driver.switch_to.window(cod_ventana_principal)
        time.sleep(5)

    except Exception as e:
        print(f"[Fila {i}] Error durante exportación: {e}")
        driver.switch_to.window(cod_ventana_principal)
        continue


<H1>TRATAMIENTO DE DATOS<H1>

<H2>Opciones</H2>
<ul>
<li>Pedir desde selenium que importe los excels a una carpeta en especifica (ruta de carpeta en una variable)</li>
<li>Cuando termine de descargar mover todos los archivos "session" a una carpeta con la libreria os y recien trabajarlos</li>
</ul>

In [12]:
def mover_descargas():

    ruta_origen = r'C:\Users\bbartolome\Downloads'
    ruta_destino = r'C:\Users\bbartolome\OneDrive - Lock & Asociados\Session descargados'

    patron_excel = 'Session_export*.xlsx'
    ruta_patron = os.path.join(ruta_origen, patron_excel)
    archivos_a_mover = glob.glob(ruta_patron)


    if not archivos_a_mover:
        print("No se encontraron archivos")
    else:
        print(f"{len(archivos_a_mover)} archivos encontrados.")
        
        if not os.path.exists(ruta_destino):
            os.makedirs(ruta_destino)
            print("carpeta creada")

        for ruta_origen_archivo in archivos_a_mover:
            
            nombre_archivo = os.path.basename(ruta_origen_archivo)
            ruta_destino_archivo = os.path.join(ruta_destino, nombre_archivo)
            
            try:
                shutil.move(ruta_origen_archivo, ruta_destino_archivo)

            except Exception as e:
                print(f"Error al mover {nombre_archivo}: {e}")
    


In [13]:
def barra_de_carga(actual, total, largo_barra=40, mensaje="Procesando..."):
    porcentaje_completo = (actual / total)
    num_caracteres_llenos = int(porcentaje_completo * largo_barra)
    caracter_lleno = '█'
    caracter_vacio = '-'

    barra = (caracter_lleno * num_caracteres_llenos) + (caracter_vacio * (largo_barra - num_caracteres_llenos))

    texto_salida = f"{mensaje} |{barra}| {int(porcentaje_completo * 100)}% ({actual}/{total})"
    
    print(texto_salida, end='\r', file=sys.stdout)

In [14]:
def subir_excel(hoja):

    carpeta = r'C:\Users\bbartolome\OneDrive - Lock & Asociados\Session descargados'
    archivos = glob.glob(os.path.join(carpeta, '*.xlsx'))
    dataframes = []

    total_archivos = len(archivos)
    
    print(f"\n--- Iniciando consolidación para la hoja: **{hoja}** ---")

    for i,archivo in enumerate(archivos):
        df = pd.read_excel(archivo, sheet_name=hoja)
        dataframes.append(df)
        barra_de_carga(actual=i + 1, total=total_archivos, mensaje=f"Leyendo archivos de '{hoja}'")
        
        df_completo = pd.concat(dataframes, ignore_index=True)
    return df_completo

In [15]:
hojas = ['IR_Actual', 'IR_InventoryPricing', 'IR_Metrics', 'IR_MQ', 'IR_Scenes', 'IR_Sessions']
lista_df = []

for hoja in hojas:
    lista_df.append(subir_excel(hoja))


--- Iniciando consolidación para la hoja: **IR_Actual** ---
Leyendo archivos de 'IR_Actual' |████████████████████████████████████████| 100% (35/35)
--- Iniciando consolidación para la hoja: **IR_InventoryPricing** ---
Leyendo archivos de 'IR_InventoryPricing' |████████████████████████████████████████| 100% (35/35)
--- Iniciando consolidación para la hoja: **IR_Metrics** ---


C:\Users\bbartolome\AppData\Local\Temp\ipykernel_25188\1145282825.py:16: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_completo = pd.concat(dataframes, ignore_index=True)
C:\Users\bbartolome\AppData\Local\Temp\ipykernel_25188\1145282825.py:16: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_completo = pd.concat(dataframes, ignore_index=True)


C:\Users\bbartolome\AppData\Local\Temp\ipykernel_25188\1145282825.py:16: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_completo = pd.concat(dataframes, ignore_index=True)


C:\Users\bbartolome\AppData\Local\Temp\ipykernel_25188\1145282825.py:16: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_completo = pd.concat(dataframes, ignore_index=True)
C:\Users\bbartolome\AppData\Local\Temp\ipykernel_25188\1145282825.py:16: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_completo = pd.concat(dataframes, ignore_index=True)


C:\Users\bbartolome\AppData\Local\Temp\ipykernel_25188\1145282825.py:16: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_completo = pd.concat(dataframes, ignore_index=True)
C:\Users\bbartolome\AppData\Local\Temp\ipykernel_25188\1145282825.py:16: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_completo = pd.concat(dataframes, ignore_index=True)


...

C:\Users\bbartolome\AppData\Local\Temp\ipykernel_25188\1145282825.py:16: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_completo = pd.concat(dataframes, ignore_index=True)
C:\Users\bbartolome\AppData\Local\Temp\ipykernel_25188\1145282825.py:16: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_completo = pd.concat(dataframes, ignore_index=True)
C:\Users\bbartolome\AppData\Local\Temp\ipykernel_25188\1145282825.py:16: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA ent

C:\Users\bbartolome\AppData\Local\Temp\ipykernel_25188\1145282825.py:16: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_completo = pd.concat(dataframes, ignore_index=True)
C:\Users\bbartolome\AppData\Local\Temp\ipykernel_25188\1145282825.py:16: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_completo = pd.concat(dataframes, ignore_index=True)
C:\Users\bbartolome\AppData\Local\Temp\ipykernel_25188\1145282825.py:16: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA ent

C:\Users\bbartolome\AppData\Local\Temp\ipykernel_25188\1145282825.py:16: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_completo = pd.concat(dataframes, ignore_index=True)
C:\Users\bbartolome\AppData\Local\Temp\ipykernel_25188\1145282825.py:16: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_completo = pd.concat(dataframes, ignore_index=True)
C:\Users\bbartolome\AppData\Local\Temp\ipykernel_25188\1145282825.py:16: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA ent

C:\Users\bbartolome\AppData\Local\Temp\ipykernel_25188\1145282825.py:16: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_completo = pd.concat(dataframes, ignore_index=True)
C:\Users\bbartolome\AppData\Local\Temp\ipykernel_25188\1145282825.py:16: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_completo = pd.concat(dataframes, ignore_index=True)
C:\Users\bbartolome\AppData\Local\Temp\ipykernel_25188\1145282825.py:16: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA ent

C:\Users\bbartolome\AppData\Local\Temp\ipykernel_25188\1145282825.py:16: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_completo = pd.concat(dataframes, ignore_index=True)


In [19]:
total_hojas = len(hojas)

for i, (hoja, nombre) in enumerate(zip(lista_df, hojas)):
    barra_de_carga(actual=i+1, total = total_hojas, mensaje=f"Guardando archivos de '{nombre}'")
    hoja.to_excel(f'C:/Users/bbartolome/OneDrive - Lock & Asociados/Gestión TI - PROYECTO LINDLEY/FDE ARCA 2025/DESCARGA SURVEY/AASS 21.11.2025 - 23.11.2025/{nombre}.xlsx', index = False)

print()
print("¡Archivos guardados con éxito!")


Guardando archivos de 'IR_Sessions' |████████████████████████████████████████| 100% (6/6)% (2/6)
¡Archivos guardados con éxito!
